<a href="https://colab.research.google.com/github/BOINI-VAMSHIKRISHNA/YOLO_PROJECT/blob/main/YOLOv7_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [ ]:
!git clone https://github.com/WongKinYiu/yolov7.git
%cd yolov7

Cloning into 'yolov7'...
remote: Enumerating objects: 1197, done.
remote: Total 1197 (delta 0), reused 0 (delta 0), pack-reused 1197 (from 1)
Receiving objects: 100% (1197/1197), 74.29 MiB | 34.64 MiB/s, done.
Resolving deltas: 100% (511/511), done.
/content/yolov7


In [ ]:
%%writefile /content/yolov7/data/custom.yaml

train: /content/yolo_dataset/images/train
val: /content/yolo_dataset/images/val

nc: 7

names: ['Person', 'Bike', 'Car', 'Helmet', 'Jacket', 'Fire', 'Smoke']

Writing /content/yolov7/data/custom.yaml


In [ ]:
!wget https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7_training.pt

--2026-05-30 06:07:01--  https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7_training.pt
Resolving github.com (github.com)... 140.82.116.3
Connecting to github.com (github.com)|140.82.116.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/511187726/13e046d1-f7f0-43ab-910b-480613181b1f?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-05-30T06%3A57%3A54Z&rscd=attachment%3B+filename%3Dyolov7_training.pt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-05-30T05%3A57%3A32Z&ske=2026-05-30T06%3A57%3A54Z&sks=b&skv=2018-11-09&sig=JxmVE68EQo0TVRkxCjcVUoyUgnGp1NT%2BWU9AOiMdQaE%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc4MDEyMzAyMSwibmJmIjoxNzgwMTIxMjIxLCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdGlvbi

In [ ]:
import re

file_path = "/content/yolov7/train.py"

with open(file_path, "r") as f:
    content = f.read()

content = content.replace(
    "torch.load(weights, map_location=device)",
    "torch.load(weights, map_location=device, weights_only=False)"
)

with open(file_path, "w") as f:
    f.write(content)

print("train.py fixed successfully!")

train.py fixed successfully!


In [ ]:
!python train.py \
--img 640 \
--batch 16 \
--epochs 80 \
--data data/custom.yaml \
--weights yolov7_training.pt \
--device 0

2026-05-30 06:07:10.532847: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
YOLOR 🚀 v0.1-128-ga207844 torch 2.11.0+cu128 CUDA:0 (Tesla T4, 14912.6875MB)

Namespace(weights='yolov7_training.pt', cfg='', data='data/custom.yaml', hyp='data/hyp.scratch.p5.yaml', epochs=80, batch_size=16, img_size=[640, 640], rect=False, resume=False, nosave=False, notest=False, noautoanchor=False, evolve=False, bucket='', cache_images=False, image_weights=False, device='0', multi_scale=False, single_cls=False, adam=False, sync_bn=False, local_rank=-1, workers=8, project='runs/train', entity=None, name='exp', exist_ok=False, quad=False, linear_lr=False, label_smoothing=0.0, upload_dataset=False, bbox_interval=-1, save_period=-1, artifact_alias='latest', freeze=[0], v5

In [ ]:
file_path = "/content/yolov7/utils/general.py"

with open(file_path, "r") as f:
    data = f.read()

data = data.replace(
    "torch.load(f, map_location=torch.device('cpu'))",
    "torch.load(f, map_location=torch.device('cpu'), weights_only=False)"
)

with open(file_path, "w") as f:
    f.write(data)

print("Fixed successfully!")

Fixed successfully!


In [ ]:
!ls /content/yolov7/runs/train

exp  exp2  exp3


In [ ]:
!ls /content/yolov7/runs/train/exp/weights

In [ ]:
from google.colab import files

files.download('/content/yolov7/runs/train/exp3/weights/best.pt')
files.download('/content/yolov7/runs/train/exp3/weights/last.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
file_path = "/content/yolov7/models/experimental.py"

with open(file_path, "r") as f:
    data = f.read()

data = data.replace(
    "ckpt = torch.load(w, map_location=map_location)",
    "ckpt = torch.load(w, map_location=map_location, weights_only=False)"
)

with open(file_path, "w") as f:
    f.write(data)

print("detect.py loading issue fixed!")

detect.py loading issue fixed!


In [ ]:
!python detect.py \
--weights /content/yolov7/runs/train/exp3/weights/best.pt \
--img-size 640 \
--conf 0.25 \
--source /content/test.jpg

Namespace(weights=['/content/yolov7/runs/train/exp3/weights/best.pt'], source='/content/test.jpg', img_size=640, conf_thres=0.25, iou_thres=0.45, device='', view_img=False, save_txt=False, save_conf=False, nosave=False, classes=None, agnostic_nms=False, augment=False, update=False, project='runs/detect', name='exp', exist_ok=False, no_trace=False)
YOLOR 🚀 v0.1-128-ga207844 torch 2.11.0+cu128 CUDA:0 (Tesla T4, 14912.6875MB)

Fusing layers... 
RepConv.fuse_repvgg_block
RepConv.fuse_repvgg_block
RepConv.fuse_repvgg_block
IDetect.fuse
Model Summary: 314 layers, 36514136 parameters, 6194944 gradients
 Convert model to Traced-model... 
 traced_script_module saved! 
 model is traced! 

/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-def

In [ ]:
file_path = "/content/yolov7/utils/datasets.py"

with open(file_path, "r") as f:
    data = f.read()

data = data.replace(
    "cache, exists = torch.load(cache_path), True",
    "cache, exists = torch.load(cache_path, weights_only=False), True"
)

with open(file_path, "w") as f:
    f.write(data)

print("datasets.py fixed successfully!")

datasets.py fixed successfully!


In [ ]:
!python test.py \
--data /content/yolov7/data/custom.yaml \
--img 640 \
--batch 16 \
--conf 0.001 \
--iou 0.65 \
--device 0 \
--weights /content/yolov7/runs/train/exp3/weights/best.pt \
--name yolov7_eval

Namespace(weights=['/content/yolov7/runs/train/exp3/weights/best.pt'], data='/content/yolov7/data/custom.yaml', batch_size=16, img_size=640, conf_thres=0.001, iou_thres=0.65, task='val', device='0', single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=False, project='runs/test', name='yolov7_eval', exist_ok=False, no_trace=False, v5_metric=False)
YOLOR 🚀 v0.1-128-ga207844 torch 2.11.0+cu128 CUDA:0 (Tesla T4, 14912.6875MB)

Fusing layers... 
RepConv.fuse_repvgg_block
RepConv.fuse_repvgg_block
RepConv.fuse_repvgg_block
IDetect.fuse
Model Summary: 314 layers, 36514136 parameters, 6194944 gradients
 Convert model to Traced-model... 
 traced_script_module saved! 
 model is traced! 

/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return 

In [ ]:
!zip -r yolov7_metrics.zip /content/yolov7/runs/test/yolov7_eval2

  adding: content/yolov7/runs/test/yolov7_eval2/ (stored 0%)
  adding: content/yolov7/runs/test/yolov7_eval2/P_curve.png (deflated 7%)
  adding: content/yolov7/runs/test/yolov7_eval2/PR_curve.png (deflated 8%)
  adding: content/yolov7/runs/test/yolov7_eval2/R_curve.png (deflated 6%)
  adding: content/yolov7/runs/test/yolov7_eval2/test_batch2_pred.jpg (deflated 6%)
  adding: content/yolov7/runs/test/yolov7_eval2/F1_curve.png (deflated 5%)
  adding: content/yolov7/runs/test/yolov7_eval2/test_batch0_pred.jpg (deflated 8%)
  adding: content/yolov7/runs/test/yolov7_eval2/test_batch0_labels.jpg (deflated 8%)
  adding: content/yolov7/runs/test/yolov7_eval2/test_batch1_labels.jpg (deflated 4%)
  adding: content/yolov7/runs/test/yolov7_eval2/test_batch2_labels.jpg (deflated 6%)
  adding: content/yolov7/runs/test/yolov7_eval2/confusion_matrix.png (deflated 22%)
  adding: content/yolov7/runs/test/yolov7_eval2/test_batch1_pred.jpg (deflated 4%)


In [ ]:
from google.colab import files
files.download("yolov7_metrics.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!python detect.py \
--weights /content/yolov7/runs/train/exp3/weights/best.pt \
--img-size 640 \
--conf 0.25 \
--source /content/smoke_person.jpg

Namespace(weights=['/content/yolov7/runs/train/exp3/weights/best.pt'], source='/content/smoke_person.jpg', img_size=640, conf_thres=0.25, iou_thres=0.45, device='', view_img=False, save_txt=False, save_conf=False, nosave=False, classes=None, agnostic_nms=False, augment=False, update=False, project='runs/detect', name='exp', exist_ok=False, no_trace=False)
YOLOR 🚀 v0.1-128-ga207844 torch 2.11.0+cu128 CUDA:0 (Tesla T4, 14912.6875MB)

Fusing layers... 
RepConv.fuse_repvgg_block
RepConv.fuse_repvgg_block
RepConv.fuse_repvgg_block
IDetect.fuse
Model Summary: 314 layers, 36514136 parameters, 6194944 gradients
 Convert model to Traced-model... 
 traced_script_module saved! 
 model is traced! 

/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[

In [ ]:
#for confidence score
!python detect.py \
--weights /content/yolov7/runs/train/exp3/weights/best.pt \
--source /content/test.jpg \
--save-txt \
--save-conf